# Article -> Country (wikilinks)
This notebook computes predicted countries based on article wikilinks.

## Setup

In [1]:
import wmfdata

In [2]:
spark = wmfdata.spark.create_session(app_name='pyspark regular; regions; isaacj',
                                  type='yarn-regular', # local, yarn-regular, yarn-large
                                  )  

SPARK_HOME: /usr/lib/spark3
Using Hadoop client lib jars at 3.2.0, provided by Spark.
PYSPARK_PYTHON=/opt/conda-analytics/bin/python3


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/07/26 14:08:50 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in mesos/standalone/kubernetes and LOCAL_DIRS in YARN).
24/07/26 14:08:50 WARN Utils: Service 'sparkDriver' could not bind on port 12000. Attempting port 12001.
24/07/26 14:08:50 WARN Utils: Service 'sparkDriver' could not bind on port 12001. Attempting port 12002.
24/07/26 14:08:50 WARN Utils: Service 'sparkDriver' could not bind on port 12002. Attempting port 12003.
24/07/26 14:08:50 WARN Utils: Service 'sparkDriver' could not bind on port 12003. Attempting port 12004.
24/07/26 14:08:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/07/26 14:08:51 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
24/07/26 14:08:51 WARN Utils: Service 'SparkUI' could n

## Compute data

In [3]:
groundtruth_table = 'isaacj.qid_to_country'
print("Groundtruth snapshots:")
spark.sql(f'SHOW PARTITIONS {groundtruth_table}').show(50, False)
spark.sql(f"DESCRIBE {groundtruth_table}").show(50, False)

Groundtruth snapshots:
+-------------------+
|partition          |
+-------------------+
|snapshot=2024-04-01|
+-------------------+

+-----------------------+---------+-----------------------------------------------------------------------------+
|col_name               |data_type|comment                                                                      |
+-----------------------+---------+-----------------------------------------------------------------------------+
|qid                    |string   |Wikidata ID of item with at least one Wikipedia sitelink -- e.g., Q42        |
|property               |string   |Wikidata property (e.g., P625 for coordinates) from which country was derived|
|country                |string   |Region name                                                                  |
|snapshot               |string   |Wikidata snapshot processed                                                  |
|# Partition Information|         |                                 

In [4]:
links_table = 'research.article_topics'
spark.sql(f'SHOW PARTITIONS {links_table}').show(50, False)
spark.sql(f"DESCRIBE {links_table}").show(50, False)

+-------------------------------------+
|partition                            |
+-------------------------------------+
|snapshot=2024-06/wiki_db=abwiki      |
|snapshot=2024-06/wiki_db=acewiki     |
|snapshot=2024-06/wiki_db=adywiki     |
|snapshot=2024-06/wiki_db=afwiki      |
|snapshot=2024-06/wiki_db=alswiki     |
|snapshot=2024-06/wiki_db=altwiki     |
|snapshot=2024-06/wiki_db=amiwiki     |
|snapshot=2024-06/wiki_db=amwiki      |
|snapshot=2024-06/wiki_db=angwiki     |
|snapshot=2024-06/wiki_db=anpwiki     |
|snapshot=2024-06/wiki_db=anwiki      |
|snapshot=2024-06/wiki_db=arcwiki     |
|snapshot=2024-06/wiki_db=arwiki      |
|snapshot=2024-06/wiki_db=arywiki     |
|snapshot=2024-06/wiki_db=arzwiki     |
|snapshot=2024-06/wiki_db=astwiki     |
|snapshot=2024-06/wiki_db=aswiki      |
|snapshot=2024-06/wiki_db=atjwiki     |
|snapshot=2024-06/wiki_db=avkwiki     |
|snapshot=2024-06/wiki_db=avwiki      |
|snapshot=2024-06/wiki_db=awawiki     |
|snapshot=2024-06/wiki_db=aywiki      |


In [10]:
groundtruth_snapshot = "2024-04-01"
links_table_snapshot = "2024-06"
wikilinks_results_table = f"isaacj.qid_to_country_wikilinks_{groundtruth_snapshot.replace('-', '_')}"
create_table_query = f"""
    CREATE TABLE IF NOT EXISTS {wikilinks_results_table} (
        pid_from         INT     COMMENT 'Page ID of Wikipedia article -- e.g., 12345',
        wiki_db          STRING  COMMENT 'Wiki -- e.g., enwiki',
        country          STRING  COMMENT 'Region name',
        tfidf            FLOAT   COMMENT 'Proportional support for country',
        count            INT     COMMENT 'Number of supporting links for country'
    )
    """

print(create_table_query)
spark.sql(create_table_query)


    CREATE TABLE IF NOT EXISTS isaacj.qid_to_country_wikilinks_2024_04_01 (
        pid_from         INT     COMMENT 'Page ID of Wikipedia article -- e.g., 12345',
        wiki_db          STRING  COMMENT 'Wiki -- e.g., enwiki',
        country          STRING  COMMENT 'Region name',
        tfidf            FLOAT   COMMENT 'Proportional support for country',
        count            INT     COMMENT 'Number of supporting links for country'
    )
    


24/07/26 14:09:22 WARN ResolveSessionCatalog: A Hive serde table will be created as there is no table provider specified. You can set spark.sql.legacy.createHiveTableByDefault to false so that native data source table will be created instead.


DataFrame[]

## Data

In [11]:
IDF = {
    'non-geo': 0.426,
    'France': 0.732,
    'United States': 0.878,
    'Italy': 1.027,
    'Mexico': 1.252,
    'Russia': 1.291,
    'United Kingdom': 1.398,
    'Germany': 1.457,
    'Spain': 1.487,
    'Japan': 1.549,
    'Iran': 1.627,
    'India': 1.68,
    'Ukraine': 1.756,
    'China': 1.766,
    'Poland': 1.772,
    'Brazil': 1.779,
    'Canada': 1.848,
    'Turkey': 1.906,
    'Romania': 1.959,
    'Czech Republic': 1.97,
    'Australia': 1.99,
    'Finland': 2.027,
    'Sweden': 2.082,
    'South Korea': 2.092,
    'Switzerland': 2.096,
    'Norway': 2.105,
    'Hungary': 2.118,
    'Belgium': 2.16,
    'Indonesia': 2.175,
    'Netherlands': 2.2,
    'Israel': 2.22,
    'Austria': 2.241,
    'Azerbaijan': 2.263,
    'Greece': 2.264,
    'Croatia': 2.271,
    'Serbia': 2.282,
    'Belarus': 2.316,
    'Slovakia': 2.331,
    'Denmark': 2.332,
    'Slovenia': 2.335,
    'South Africa': 2.348,
    'Ireland': 2.366,
    'Malaysia': 2.38,
    'Syria': 2.384,
    'Philippines': 2.398,
    'Estonia': 2.419,
    'Argentina': 2.427,
    'Portugal': 2.431,
    'Pakistan': 2.436,
    'New Zealand': 2.436,
    'Yemen': 2.458,
    'Bulgaria': 2.459,
    'Kazakhstan': 2.465,
    'Georgia': 2.5,
    'Bosnia and Herzegovina': 2.514,
    'Latvia': 2.523,
    'Taiwan': 2.538,
    'Lithuania': 2.56,
    'Egypt': 2.58,
    'Lebanon': 2.581,
    'Armenia': 2.627,
    'Morocco': 2.635,
    'Cameroon': 2.644,
    'Nigeria': 2.652,
    'Thailand': 2.656,
    'Cambodia': 2.658,
    'Vietnam': 2.662,
    'Kenya': 2.663,
    'Colombia': 2.667,
    'Singapore': 2.668,
    'Algeria': 2.673,
    'United Arab Emirates': 2.674,
    'Oman': 2.68,
    'Peru': 2.683,
    'Moldova': 2.684,
    'Madagascar': 2.69,
    'Sri Lanka': 2.698,
    'Tanzania': 2.702,
    'Bangladesh': 2.706,
    'Malta': 2.712,
    'Chile': 2.725,
    'Bahrain': 2.734,
    'Sudan': 2.744,
    'Uganda': 2.752,
    'Rwanda': 2.752,
    'Ghana': 2.755,
    'Mauritius': 2.76,
    'Ethiopia': 2.762,
    'Papua New Guinea': 2.763,
    'Uzbekistan': 2.766,
    'Jamaica': 2.767,
    'Vanuatu': 2.769,
    'Saudi Arabia': 2.774,
    'Belize': 2.775,
    'Seychelles': 2.784,
    'Iraq': 2.785,
    'Aruba': 2.786,
    'Guernsey': 2.787,
    'Jersey': 2.788,
    'Gibraltar': 2.79,
    'Tunisia': 2.797,
    'Luxembourg': 2.8,
    'Eritrea': 2.8,
    'North Macedonia': 2.804,
    'Fiji': 2.811,
    'Zambia': 2.812,
    'South Sudan': 2.815,
    'Brunei': 2.817,
    'Zimbabwe': 2.819,
    'Bhutan': 2.819,
    'Afghanistan': 2.821,
    'Trinidad and Tobago': 2.823,
    'Maldives': 2.832,
    'Namibia': 2.833,
    'Tajikistan': 2.834,
    'Guyana': 2.834,
    'Botswana': 2.835,
    'Liberia': 2.848,
    'Eswatini': 2.853,
    'Burkina Faso': 2.854,
    'Solomon Islands': 2.854,
    'Bahamas': 2.855,
    'Samoa': 2.856,
    'Hong Kong': 2.856,
    'Sierra Leone': 2.86,
    'Antigua and Barbuda': 2.861,
    'Malawi': 2.861,
    'Saint Lucia': 2.862,
    'Barbados': 2.863,
    'Kyrgyzstan': 2.865,
    'Isle of Man': 2.865,
    'Gambia': 2.867,
    'Lesotho': 2.867,
    'Nepal': 2.868,
    'Saint Vincent and the Grenadines': 2.87,
    'Grenada': 2.87,
    'Federated States of Micronesia': 2.87,
    'Palau': 2.871,
    'Dominica': 2.872,
    'Tonga': 2.877,
    'Saint Kitts and Nevis': 2.877,
    'Marshall Islands': 2.881,
    'Kiribati': 2.882,
    'American Samoa': 2.883,
    'Bermuda': 2.884,
    'Nauru': 2.884,
    'Antarctica': 2.884,
    'Tuvalu': 2.885,
    'Myanmar': 2.887,
    'Sint Maarten': 2.888,
    'Cook Islands': 2.888,
    'Saint Helena, Ascension, and Tristan da Cunha': 2.889,
    'Albania': 2.892,
    'United States Virgin Islands': 2.893,
    'British Virgin Islands': 2.893,
    'Falkland Islands': 2.894,
    'Anguilla': 2.898,
    'Cayman Islands': 2.898,
    'Niue': 2.9,
    'Montserrat': 2.901,
    'Pitcairn Islands': 2.901,
    'Turks and Caicos Islands': 2.902,
    'Saint Martin': 2.907,
    'Tokelau': 2.907,
    'British Indian Ocean Territory': 2.909,
    'Bolivia': 2.931,
    'Venezuela': 2.936,
    'Uruguay': 2.943,
    'Montenegro': 2.943,
    'Palestine': 2.948,
    'Vatican City': 2.959,
    'Cyprus': 2.963,
    'Ecuador': 2.997,
    'Mali': 2.999,
    'Andorra': 3.002,
    'Mongolia': 3.01,
    'Monaco': 3.03,
    'Democratic Republic of the Congo': 3.037,
    'Senegal': 3.053,
    'Cuba': 3.055,
    'Niger': 3.09,
    'Iceland': 3.1,
    'Panama': 3.106,
    'Costa Rica': 3.107,
    'Jordan': 3.112,
    'Mauritania': 3.117,
    'Guatemala': 3.14,
    'Laos': 3.141,
    'Paraguay': 3.141,
    'Dominican Republic': 3.142,
    'North Korea': 3.142,
    'Turkmenistan': 3.148,
    'Chad': 3.151,
    'Kosovo': 3.153,
    'Ivory Coast': 3.164,
    'Angola': 3.166,
    'Equatorial Guinea': 3.166,
    'Liechtenstein': 3.168,
    'Haiti': 3.189,
    'Kuwait': 3.195,
    'Libya': 3.196,
    'Togo': 3.197,
    'Qatar': 3.197,
    'Honduras': 3.198,
    'Benin': 3.223,
    'Djibouti': 3.225,
    'Comoros': 3.239,
    'Guinea': 3.243,
    'Republic of the Congo': 3.245,
    'San Marino': 3.258,
    'El Salvador': 3.268,
    'Gabon': 3.27,
    'Central African Republic': 3.276,
    'Nicaragua': 3.3,
    'Burundi': 3.309,
    'East Timor': 3.33,
    'Western Sahara': 3.356,
    'Somalia': 3.375,
    'Mozambique': 3.397,
    'Puerto Rico': 3.481,
    'United States Minor Outlying Islands': 3.539,
    'Suriname': 3.54,
    'Cape Verde': 3.639,
    'Greenland': 3.679,
    'Macao': 3.698,
    'Guinea-Bissau': 3.753,
    'Faroe Islands': 3.756,
    'São Tomé and Príncipe': 3.761,
    'Curaçao': 3.85,
    'Guam': 4.072,
    'Bonaire, Sint Eustatius, and Saba': 4.133,
    'Réunion': 4.143,
    'Åland': 4.16,
    'Guadeloupe': 4.174,
    'French Polynesia': 4.18,
    'New Caledonia': 4.189,
    'Martinique': 4.206,
    'French Guiana': 4.23,
    'South Georgia and the South Sandwich Islands': 4.325,
    'Saint Pierre and Miquelon': 4.482,
    'Northern Mariana Islands': 4.494,
    'Mayotte': 4.515,
    'French Southern and Antarctic Lands': 4.516,
    'Wallis and Futuna': 4.715,
    'Norfolk Island': 4.736,
    'Saint Barthélemy': 4.908,
    'Cocos (Keeling) Islands': 4.977,
    'Christmas Island': 4.983,
    'Bouvet Island': 5.013,
}

## Functions

In [12]:
def getCountryPredictions(link_countries):
    try:
        country_counts = {}
        for c in link_countries:
            country_counts[c] = country_counts.get(c, 0) + 1
        candidates = []
        results = []
        links_analyzed = len(link_countries)
        tfidf_sum = 0
        for c, count in country_counts.items():
            try:
                prop_tfidf = (count / links_analyzed) * IDF[c]
                tfidf_sum +=  prop_tfidf
                if c != 'non-geo' and count > 1:
                    candidates.append((c, count, prop_tfidf))
            except Exception:
                continue
        for c in candidates:
            normalized_tfidf = c[2] / tfidf_sum
            if normalized_tfidf >= 0.05:
                results.append((c[0], normalized_tfidf, c[1]))
        return results
    except Exception:
        return None
    
spark.udf.register('getCountryPredictions', getCountryPredictions,
                   'Array<Struct<country:String, tfidf:Float, count:Int>>')

24/07/26 14:09:38 WARN SimpleFunctionRegistry: The function getcountrypredictions replaced a previously registered function.


<function __main__.getCountryPredictions(link_countries)>

In [13]:
query = f"""
WITH groundtruth AS (
    SELECT DISTINCT
      qid,
      country
    FROM {groundtruth_table}
    WHERE
      snapshot = '{groundtruth_snapshot}'
),
pagelinks AS (
    SELECT
      wiki_db,
      pid_from,
      EXPLODE(SPLIT(outlinks, " ")) AS outlink
    FROM {links_table}
    WHERE
      snapshot = '{links_table_snapshot}'
),
article_to_countries AS (
    SELECT
      wiki_db,
      pid_from,
      COLLECT_LIST(COALESCE(country, "non-geo")) AS countries
    FROM pagelinks pl
    LEFT JOIN groundtruth gt
      ON (pl.outlink = gt.qid)
    GROUP BY
      wiki_db,
      pid_from
)
INSERT INTO TABLE {wikilinks_results_table}
SELECT
  pid_from,
  wiki_db,
  INLINE(getCountryPredictions(countries))
FROM article_to_countries
"""

print(query)
spark.sql(query)


WITH groundtruth AS (
    SELECT DISTINCT
      qid,
      country
    FROM isaacj.qid_to_country
    WHERE
      snapshot = '2024-04-01'
),
pagelinks AS (
    SELECT
      wiki_db,
      pid_from,
      EXPLODE(SPLIT(outlinks, " ")) AS outlink
    FROM research.article_topics
    WHERE
      snapshot = '2024-06'
),
article_to_countries AS (
    SELECT
      wiki_db,
      pid_from,
      COLLECT_LIST(COALESCE(country, "non-geo")) AS countries
    FROM pagelinks pl
    LEFT JOIN groundtruth gt
      ON (pl.outlink = gt.qid)
    GROUP BY
      wiki_db,
      pid_from
)
INSERT INTO TABLE isaacj.qid_to_country_wikilinks_2024_04_01
SELECT
  pid_from,
  wiki_db,
  INLINE(getCountryPredictions(countries))
FROM article_to_countries



DataFrame[]

In [14]:
# check output (NEW)
# NOTE: this was done after the cultural ones had already been
#       added to the table so the output reflects that too.
spark.sql(f"""
SELECT
  country,
  COUNT(1) AS num_results
FROM {wikilinks_results_table}
WHERE
  tfidf >= 0.25
  AND count >= 3
GROUP BY
  country
ORDER BY
  num_results DESC
""").show(500, False)


+---------------------------------------------+-----------+
|country                                      |num_results|
+---------------------------------------------+-----------+
|United States                                |5412961    |
|France                                       |2099308    |
|Germany                                      |1888531    |
|United Kingdom                               |1756852    |
|Japan                                        |1558746    |
|Italy                                        |1229742    |
|Spain                                        |1041347    |
|Russia                                       |1036440    |
|India                                        |934519     |
|Poland                                       |910735     |
|Norway                                       |757182     |
|Canada                                       |737858     |
|China                                        |719394     |
|Mexico                                 